<a href="https://colab.research.google.com/github/shaikharaina/Storm-/blob/main/Rahat_NOAA_Storm_Processing_Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
# 1. Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
DETAILES_CSV = '/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_details.csv'
FATALITIES_CSV = '/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_fatalities.csv'
LOCATIONS_CSV = '/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_locations.csv'

In [4]:
import pandas as pd
import numpy as np
import glob
import os

In [5]:
def clean_damage(value):
    """Converts K, M, B suffixes to numeric floats."""
    if pd.isna(value) or value == 0:
        return 0.0
    value = str(value).upper().strip()
    multiplier = 1.0
    if 'K' in value:
        multiplier = 1e3
    elif 'M' in value:
        multiplier = 1e6
    elif 'B' in value:
        multiplier = 1e9

    # Remove non-numeric characters except decimal points
    try:
        clean_num = ''.join(c for c in value if c.isdigit() or c == '.')
        return float(clean_num) * multiplier
    except:
        return 0.0


def cluster_events(event):
    event = str(event).upper()
    if any(x in event for x in ['TORNADO', 'FUNNEL']): return 'Convective_Rotational'
    if any(x in event for x in ['THUNDERSTORM', 'WIND', 'HAIL', 'LIGHTNING']): return 'Convective_Severe'
    if any(x in event for x in ['FLOOD', 'RAIN']): return 'Hydrological'
    if any(x in event for x in ['SNOW', 'ICE', 'BLIZZARD', 'WINTER', 'COLD', 'FREEZE']): return 'Winter'
    if any(x in event for x in ['HEAT', 'DROUGHT']): return 'Heat_Drought'
    if any(x in event for x in ['MARINE', 'WATERSPOUT', 'SURF', 'TIDE']): return 'Marine'
    return 'Other'


f_scale_mapping = {
    'EF0': 0, 'F0': 0,
    'EF1': 1, 'F1': 1,
    'EF2': 2, 'F2': 2,
    'EF3': 3, 'F3': 3,
    'EF4': 4, 'F4': 4,
    'EF5': 5, 'F5': 5
}

In [7]:
import os

archive_path = '/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive'

for root, dirs, files in os.walk(archive_path):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_fatalities.csv/StormEvents_fatalities.csv
/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_locations.csv/StormEvents_locations.csv
/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_details.csv/StormEvents_details.csv


In [9]:
DETAILES_CSV = '/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_details.csv/StormEvents_details.csv'

FATALITIES_CSV = '/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_fatalities.csv/StormEvents_fatalities.csv'

LOCATIONS_CSV = '/content/drive/MyDrive/RP_Training_Tabular/Dataset/archive/StormEvents_locations.csv/StormEvents_locations.csv'

In [10]:
import pandas as pd

df_details = pd.read_csv(DETAILES_CSV)
df_fatalities = pd.read_csv(FATALITIES_CSV)

print("Details shape:", df_details.shape)
print("Fatalities shape:", df_fatalities.shape)

/tmp/ipykernel_967/2642944303.py:3: DtypeWarning: Columns (16,25,26,28,29,34,35,37,39,40,42,43,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df_details = pd.read_csv(DETAILES_CSV)


Details shape: (1962332, 51)
Fatalities shape: (24426, 11)


In [11]:
print("Details columns:")
print(df_details.columns.tolist())

print("\nFatalities columns:")
print(df_fatalities.columns.tolist())

Details columns:
['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID', 'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY', 'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE']

Fatalities columns:
['FAT_YEARMONTH', 'FAT_DAY', 'FAT_TIME', 'FATALITY_ID', 'EVENT_ID', 'FATALITY_TYPE', 'FATALITY_DATE', 'FATALITY_AGE', 'FATALITY_SEX', 'FATALITY_LOCATION', 'EVENT_YEARMONTH']


In [12]:
import re
import pandas as pd

def clean_damage(value):
    if pd.isna(value):
        return 0

    value = str(value).strip().upper()

    if value == '' or value == '0':
        return 0

    multipliers = {
        'K': 1_000,
        'M': 1_000_000,
        'B': 1_000_000_000
    }

    last_char = value[-1]

    if last_char in multipliers:
        try:
            number = float(value[:-1])
            return number * multipliers[last_char]
        except ValueError:
            return 0

    try:
        return float(value)
    except ValueError:
        return 0

In [13]:
# 1. Load the data
df_details = pd.read_csv(DETAILES_CSV)
df_fatalities = pd.read_csv(FATALITIES_CSV, low_memory=False)

# 2. Pre-aggregate Fatalities
fatality_summary = df_fatalities.groupby('EVENT_ID').agg({
    'FATALITY_ID': 'count',
    'FATALITY_AGE': 'mean'
}).rename(
    columns={
        'FATALITY_ID': 'FATALITY_COUNT',
        'FATALITY_AGE': 'AVG_FATALITY_AGE'
    }
).reset_index()

# 3. Merge Details with Fatality Summary
master_df = pd.merge(
    df_details,
    fatality_summary,
    on='EVENT_ID',
    how='left'
)

# 4. Fill missing values for events with no fatalities
master_df['FATALITY_COUNT'] = master_df['FATALITY_COUNT'].fillna(0)

# 5. Clean Damage Columns
master_df['DAMAGE_PROPERTY_NUM'] = master_df['DAMAGE_PROPERTY'].apply(clean_damage)
master_df['DAMAGE_CROPS_NUM'] = master_df['DAMAGE_CROPS'].apply(clean_damage)

# 6. Convert Time to DateTime Objects
master_df['BEGIN_DATE_TIME'] = pd.to_datetime(
    master_df['BEGIN_DATE_TIME'],
    errors='coerce'
)

master_df['END_DATE_TIME'] = pd.to_datetime(
    master_df['END_DATE_TIME'],
    errors='coerce'
)

# 7. Feature Engineering: Storm Duration
master_df['DURATION_SECONDS'] = (
    master_df['END_DATE_TIME'] -
    master_df['BEGIN_DATE_TIME']
).dt.total_seconds()

# 8. Check the result
print(f"Master Dataset Shape: {master_df.shape}")

print(
    master_df[
        [
            'EVENT_ID',
            'EVENT_TYPE',
            'DAMAGE_PROPERTY_NUM',
            'FATALITY_COUNT',
            'DURATION_SECONDS'
        ]
    ].head()
)

/tmp/ipykernel_967/2607492962.py:2: DtypeWarning: Columns (16,25,26,28,29,34,35,37,39,40,42,43,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  df_details = pd.read_csv(DETAILES_CSV)


Master Dataset Shape: (1962332, 56)
   EVENT_ID         EVENT_TYPE  DAMAGE_PROPERTY_NUM  FATALITY_COUNT  \
0  10151721  Thunderstorm Wind                  0.0             0.0   
1  10151722  Thunderstorm Wind                  0.0             0.0   
2  10151723  Thunderstorm Wind                  0.0             0.0   
3  10132975  Thunderstorm Wind                  0.0             0.0   
4  10032952            Tornado              25000.0             0.0   

   DURATION_SECONDS  
0               0.0  
1               0.0  
2               0.0  
3               0.0  
4               0.0  


In [14]:
cols_to_drop = ['DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'CATEGORY']
# Note: 'CATEGORY' is often empty/unknown and causes issues too [cite: 70]

master_df = master_df.drop(columns=[c for c in cols_to_drop if c in master_df.columns])

# Ensure all column names are strings (Parquet requirement)
master_df.columns = master_df.columns.astype(str)

# Using the NWS Directive 10-1605 standard for EF scales
master_df['TOR_F_SCALE_NUM'] = master_df['TOR_F_SCALE'].map(f_scale_mapping).fillna(-1)

master_df['EVENT_GROUP'] = master_df['EVENT_TYPE'].apply(cluster_events)

# 3. Time-of-Day Feature (LST)
# Identifying if the storm hit during the night (higher fatality risk)
master_df['BEGIN_HOUR'] = pd.to_datetime(master_df['BEGIN_DATE_TIME']).dt.hour
master_df['IS_NIGHT'] = master_df['BEGIN_HOUR'].apply(lambda x: 1 if (x >= 20 or x <= 6) else 0)

# 4. Target Transformation: Log-Scaling Damage
# Economic damage is highly skewed; log-scaling is standard for IEEE regression
master_df['LOG_DAMAGE_PROPERTY'] = np.log1p(master_df['DAMAGE_PROPERTY_NUM'])

In [15]:
master_df.isnull().sum()

,0
BEGIN_YEARMONTH,0
BEGIN_DAY,0
BEGIN_TIME,0
END_YEARMONTH,0
END_DAY,0
END_TIME,0
EPISODE_ID,232245
EVENT_ID,0
STATE,1
STATE_FIPS,1


In [16]:
# 1. Prepare Target for Classifier (1 if damage > 0, else 0)
master_df['HAS_DAMAGE'] = (master_df['DAMAGE_PROPERTY_NUM'] > 0).astype(int)

In [17]:
from sklearn.model_selection import train_test_split


train_data, test_data = train_test_split(
    master_df,
    test_size = 0.2,
    stratify=master_df["HAS_DAMAGE"],
    random_state = 42
    )

In [18]:
import pickle


def preprocessing_model_saver(model, path_name):
    try:
        with open(path_name, "wb") as model_file:
            pickle.dump(model, model_file)

        return True
    except Exception as error:
        print("Error generated while saving the model: {}".format(str(error)))

        return False

In [20]:
import os

ENCODER_PATH = '/content/drive/MyDrive/RP_Training_Tabular/Models/ohe.pkl'

os.makedirs(
    '/content/drive/MyDrive/RP_Training_Tabular/Models',
    exist_ok=True
)

preprocessing_model_saver(ohe, ENCODER_PATH)

True

In [21]:
print("train_data exists:", 'train_data' in globals())
print("test_data exists:", 'test_data' in globals())
print("EVENT_GROUP exists:", 'EVENT_GROUP' in train_data.columns if 'train_data' in globals() else False)

train_data exists: True
test_data exists: True
EVENT_GROUP exists: True


In [22]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

category_columns = ["EVENT_GROUP"]

# Create OneHotEncoder
ohe = OneHotEncoder(
    drop="first",
    sparse_output=False
)

# Fit encoder ONLY on training data
ohe.fit(train_data[category_columns])

# Transform train and test data
train_transform = ohe.transform(train_data[category_columns])
test_transform = ohe.transform(test_data[category_columns])

# Get generated feature names
feature_names = ohe.get_feature_names_out()

# Convert transformed data into DataFrames
train_transform_df = pd.DataFrame(
    train_transform,
    columns=feature_names,
    index=train_data.index
)

test_transform_df = pd.DataFrame(
    test_transform,
    columns=feature_names,
    index=test_data.index
)

# Combine encoded columns with original data
train_df = pd.concat(
    [train_transform_df, train_data],
    axis=1
)

test_df = pd.concat(
    [test_transform_df, test_data],
    axis=1
)

# Save the encoder
preprocessing_model_saver(ohe, ENCODER_PATH)

print("Encoder saved successfully!")
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Encoded features:", feature_names)

Encoder saved successfully!
Train shape: (1569865, 65)
Test shape: (392467, 65)
Encoded features: ['EVENT_GROUP_Convective_Severe' 'EVENT_GROUP_Heat_Drought'
 'EVENT_GROUP_Hydrological' 'EVENT_GROUP_Marine' 'EVENT_GROUP_Other'
 'EVENT_GROUP_Winter']


In [24]:
print("features" in globals())

False


In [25]:
print(train_df.columns.tolist())

['EVENT_GROUP_Convective_Severe', 'EVENT_GROUP_Heat_Drought', 'EVENT_GROUP_Hydrological', 'EVENT_GROUP_Marine', 'EVENT_GROUP_Other', 'EVENT_GROUP_Winter', 'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID', 'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE', 'FATALITY_COUNT', 'AVG_FATALITY_AGE', 'DAMAGE_PROPERTY_NUM', 'DAMAGE_CROPS_NUM', 'DURATION_SECONDS', 'TOR_F_SCALE_NUM', 'EVENT_GROU

In [28]:
print(train_df.head())

         EVENT_GROUP_Convective_Severe  EVENT_GROUP_Heat_Drought  \
1890248                            0.0                       0.0   
855981                             0.0                       0.0   
1108030                            1.0                       0.0   
145041                             1.0                       0.0   
1536920                            0.0                       1.0   

         EVENT_GROUP_Hydrological  EVENT_GROUP_Marine  EVENT_GROUP_Other  \
1890248                       0.0                 0.0                0.0   
855981                        0.0                 0.0                0.0   
1108030                       0.0                 0.0                0.0   
145041                        0.0                 0.0                0.0   
1536920                       0.0                 0.0                0.0   

         EVENT_GROUP_Winter  BEGIN_YEARMONTH  BEGIN_DAY  BEGIN_TIME  \
1890248                 1.0           202502         11        

In [29]:
print(train_df.dtypes)

EVENT_GROUP_Convective_Severe    float64
EVENT_GROUP_Heat_Drought         float64
EVENT_GROUP_Hydrological         float64
EVENT_GROUP_Marine               float64
EVENT_GROUP_Other                float64
                                  ...   
EVENT_GROUP                       object
BEGIN_HOUR                         int32
IS_NIGHT                           int64
LOG_DAMAGE_PROPERTY              float64
HAS_DAMAGE                         int64
Length: 65, dtype: object


In [30]:
features = [
    'MAGNITUDE', 'TOR_F_SCALE_NUM', 'TOR_LENGTH', 'TOR_WIDTH',
    'BEGIN_LAT', 'BEGIN_LON', 'DURATION_SECONDS', 'IS_NIGHT'
] + list(feature_names) + ["HAS_DAMAGE"]
print(features)

['MAGNITUDE', 'TOR_F_SCALE_NUM', 'TOR_LENGTH', 'TOR_WIDTH', 'BEGIN_LAT', 'BEGIN_LON', 'DURATION_SECONDS', 'IS_NIGHT', 'EVENT_GROUP_Convective_Severe', 'EVENT_GROUP_Heat_Drought', 'EVENT_GROUP_Hydrological', 'EVENT_GROUP_Marine', 'EVENT_GROUP_Other', 'EVENT_GROUP_Winter', 'HAS_DAMAGE']


In [31]:
# select columns
train_df = train_df[features]
test_df = test_df[features]

In [32]:
# drop nulls
train_df_final = train_df.dropna(how = "any")
test_df_final = test_df.dropna(how = "any")

In [35]:
import os

ARTIFACTS_PATH = '/content/drive/MyDrive/RP_Training_Tabular/Artifacts'

os.makedirs(ARTIFACTS_PATH, exist_ok=True)

TRAIN_CSV = os.path.join(ARTIFACTS_PATH, 'train.csv')
TEST_CSV = os.path.join(ARTIFACTS_PATH, 'test.csv')

train_df_final.to_csv(TRAIN_CSV, index=False)
test_df_final.to_csv(TEST_CSV, index=False)

print("Train file saved at:")
print(TRAIN_CSV)

print("\nTest file saved at:")
print(TEST_CSV)

Train file saved at:
/content/drive/MyDrive/RP_Training_Tabular/Artifacts/train.csv

Test file saved at:
/content/drive/MyDrive/RP_Training_Tabular/Artifacts/test.csv


In [36]:
# rese indexes
train_df_final.reset_index(drop = True, inplace = True)
test_df_final.reset_index(drop = True, inplace = True)

In [37]:
train_df_final.to_csv(TRAIN_CSV, index = False)
test_df_final.to_csv(TEST_CSV, index = False)